# ZuCo sentiment from classical EEG features

Classical stats of the raw EEG plus ZuCo's precomputed band-power means, then a
linear classifier. A plain CPU runtime is fine here, no GPU needed.

Results (numbers) go to Drive under `RESULTS_DIR` (tagged by `RUN_TAG` so each
code change gets its own folder). Plots are written to a local, throwaway
`FIGS_DIR` just for viewing -- they're regenerable from the saved numbers.

In [ ]:
# clone (or pull) the code, install deps
import os
REPO = 'zuco-eeg-classical-features'
if not os.path.exists(REPO):
    !git clone https://github.com/parmisbathaeiyan/zuco-eeg-classical-features.git $REPO
else:
    !cd $REPO && git pull --ff-only
%cd $REPO
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# folder with the EEG results*_SR.mat files
MAT_DIR = '/content/drive/MyDrive/Thesis/Data/zuco_og_raw'

# numbers (json/csv) -> Drive, versioned by RUN_TAG. Bump RUN_TAG when you change
# the pipeline so old runs are kept side by side (e.g. 'v2_meanfeatures').
RESULTS_BASE = '/content/drive/MyDrive/Thesis/Results/zuco_eeg_classical_features'
RUN_TAG = 'v1_perchannel'
RESULTS_DIR = f'{RESULTS_BASE}/{RUN_TAG}'

# plots -> local scratch (ephemeral), just for viewing; not saved to Drive
FIGS_DIR = 'figs'

In [ ]:
# 1. extract features -> features/<SUBJ>.npz  (slow; resumable, skips cached)
!python extract_features.py --mat-dir "$MAT_DIR" --out-dir features

In [ ]:
# 2. classify: numbers -> RESULTS_DIR, plots -> FIGS_DIR
!python run.py --features-dir features --output-dir "$RESULTS_DIR" --plots-dir "$FIGS_DIR" --classifier logreg

In [ ]:
# 3. tables + report -> RESULTS_DIR/tables ; show the rendered report
!python make_tables.py --results-dir "$RESULTS_DIR" > /dev/null

from IPython.display import Markdown, display
display(Markdown(open(os.path.join(RESULTS_DIR, 'tables', 'report.md')).read()))

In [ ]:
# the classification plots (from FIGS_DIR)
from IPython.display import Image, display
for name in ['subject_accuracy_logreg.png', 'cm_pooled_logreg.png']:
    p = os.path.join(FIGS_DIR, name)
    if os.path.exists(p):
        display(Image(p))

In [ ]:
# 4. feature-label association: pooled + per-subject. numbers -> RESULTS_DIR, bars -> FIGS_DIR
!python analyze_features.py --features-dir features --output-dir "$RESULTS_DIR" --plots-dir "$FIGS_DIR" > /dev/null

from IPython.display import Markdown, Image, display
A = os.path.join(RESULTS_DIR, 'associations')
display(Markdown(open(os.path.join(A, 'associations.md')).read()))      # pooled
display(Markdown(open(os.path.join(A, 'subject_report.md')).read()))    # per-subject
for name in ['by_stat.png', 'by_band.png', 'top_features.png', 'subject_by_stat.png']:
    p = os.path.join(FIGS_DIR, name)
    if os.path.exists(p):
        display(Image(p))

In [ ]:
# 5. spatial montage: per-channel association on the scalp, one panel per family.
#    reads feature_association.csv (RESULTS_DIR), writes the topomaps to FIGS_DIR
!python plot_montage.py --results-dir "$RESULTS_DIR" --montage zuco_montage.npz --out-dir "$FIGS_DIR/montage" > /dev/null

from IPython.display import Image, display
for name in ['montage_stats_f_score.png', 'montage_bands_f_score.png']:
    p = os.path.join(FIGS_DIR, 'montage', name)
    if os.path.exists(p):
        display(Image(p))

In [ ]:
# (optional) channel-averaged comparison: 24 family means instead of 2520 per-channel.
# Tests whether averaging over channels lifts a diffuse signal out of the noise.
CHAVG_DIR = f'{RESULTS_DIR}_chavg'
!python run.py --features-dir features --output-dir "$CHAVG_DIR" --classifier logreg --channel-avg --no-plots > /dev/null
!python make_tables.py --results-dir "$CHAVG_DIR" > /dev/null
!python analyze_features.py --features-dir features --output-dir "$CHAVG_DIR" --channel-avg > /dev/null

from IPython.display import Markdown, display
display(Markdown('## Channel-averaged (24 features)'))
display(Markdown(open(os.path.join(CHAVG_DIR, 'tables', 'report.md')).read()))
display(Markdown(open(os.path.join(CHAVG_DIR, 'associations', 'associations.md')).read()))